Cellule 0  — Imports

In [1]:
# Cellule 0 — Imports
import pandas as pd
import os, json, numpy as np
from collections import defaultdict
import torch

# Vérifier sentence-transformers (installé hier)
from sentence_transformers import SentenceTransformer
print("✅ Imports OK")
print(f"CUDA : {torch.cuda.is_available()}")

✅ Imports OK
CUDA : True


Cellule 1  — Charger 72K prérequis + noms concepts

In [2]:
import pandas as pd, os, json, numpy as np
from collections import defaultdict

KG_SAMPLED = r'D:\Memoire\LLMKGER\kg_sampled'

# 1. Charger les 72,361 relations prerequisite_of du KG KGE
print("Chargement des prérequis du KG KGE...")
prereq_kge_list = []
with open(os.path.join(KG_SAMPLED, 'triplets_train_kg.tsv'), 'r', encoding='utf-8') as f:
    next(f)  # header
    for line in f:
        parts = line.rstrip().split('\t')
        if len(parts) >= 3 and parts[1] == 'prerequisite_of':
            prereq_kge_list.append((parts[0], parts[2]))

prereq_kge = pd.DataFrame(prereq_kge_list, columns=['head','tail'])
print(f"Relations prerequisite_of : {len(prereq_kge):,}")

concepts_kge = set(prereq_kge['head']) | set(prereq_kge['tail'])
print(f"Concepts uniques : {len(concepts_kge):,}")

print("\nChargement des noms de concepts...")
concept_name = {}
with open(r'D:\Memoire\mooccubeX\translated\concept_translated.json', 'r', encoding='utf-8') as f:
    for line in f:
        try:
            obj = json.loads(line)
            cid = obj.get('id','')
            name = obj.get('name','').strip()
            if cid: concept_name[cid] = name
        except: continue
print(f"Noms chargés : {len(concept_name):,}")

matched = sum(1 for c in concepts_kge if concept_name.get(c,'').strip())
print(f"Concepts KGE avec nom : {matched:,}/{len(concepts_kge):,} ({100*matched/len(concepts_kge):.1f}%)")

Chargement des prérequis du KG KGE...
Relations prerequisite_of : 72,361
Concepts uniques : 28,779

Chargement des noms de concepts...
Noms chargés : 637,568
Concepts KGE avec nom : 28,777/28,779 (100.0%)


Cellule 2  — Embeddings sémantiques des concepts

In [3]:
from sentence_transformers import SentenceTransformer
import numpy as np, time

# Charger le modèle sémantique (comme hier)
print("Chargement du modèle sémantique...")
model_sem = SentenceTransformer('all-MiniLM-L6-v2')

# Concepts à encoder (ceux avec un nom)
concepts_list = sorted([c for c in concepts_kge if concept_name.get(c,'').strip()])
names = [concept_name[c] for c in concepts_list]
print(f"Concepts à encoder : {len(concepts_list):,}")

# Calculer les embeddings
print("Calcul des embeddings...")
t0 = time.time()
concept_emb = model_sem.encode(names, batch_size=128, show_progress_bar=True,
                                convert_to_numpy=True, normalize_embeddings=True)
print(f"✅ Embeddings calculés en {(time.time()-t0)/60:.1f} min")
print(f"   Shape : {concept_emb.shape}")

concept_to_idx = {c: i for i, c in enumerate(concepts_list)}
print(f"   {len(concept_to_idx):,} concepts indexés")

Chargement du modèle sémantique...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Concepts à encoder : 28,777
Calcul des embeddings...


Batches:   0%|          | 0/225 [00:00<?, ?it/s]

✅ Embeddings calculés en 0.1 min
   Shape : (28777, 384)
   28,777 concepts indexés


Cellule 3 — calcul des critères sémantique et directionnel sur les 72 361 relations

In [4]:
import numpy as np, pandas as pd, os
from collections import defaultdict

# ===== CRITÈRE 2 : Similarité sémantique =====
print("Calcul du critère sémantique...")
sem_scores = []
for h, t in prereq_kge[['head','tail']].itertuples(index=False):
    ih = concept_to_idx.get(h)
    it = concept_to_idx.get(t)
    if ih is not None and it is not None:
        sim = float(np.dot(concept_emb[ih], concept_emb[it]))
    else:
        sim = np.nan
    sem_scores.append(sim)
prereq_kge['sem_sim'] = sem_scores
print(f"  Similarité calculée pour {prereq_kge['sem_sim'].notna().sum():,} relations")

# ===== CRITÈRE 3 : Directionnel (généralité) =====
# Charger concept -> cours (pour compter la généralité)
print("\nChargement concept-course...")
cc = pd.read_csv(r'D:\Memoire\mooccubeX\concept-course.txt', sep='\t', header=None,
                 names=['concept','course'], dtype=str)
courses_of_concept = defaultdict(set)
for concept, course in cc.itertuples(index=False):
    courses_of_concept[str(concept)].add(str(course))
print(f"  Concepts avec cours : {len(courses_of_concept):,}")

print("Calcul du critère directionnel...")
dir_scores = []
for h, t in prereq_kge[['head','tail']].itertuples(index=False):
    n_h = len(courses_of_concept.get(h, set()))
    n_t = len(courses_of_concept.get(t, set()))
    if n_h > 0 and n_t > 0:
        dir_score = (n_h - n_t) / (n_h + n_t)
    else:
        dir_score = np.nan
    dir_scores.append(dir_score)
prereq_kge['dir_score'] = dir_scores
print(f"  Score directionnel calculé pour {prereq_kge['dir_score'].notna().sum():,} relations")

# Distributions
print("\n=== Distributions ===")
vs = prereq_kge['sem_sim'].dropna()
print(f"Sémantique : mean={vs.mean():.3f}, médiane={vs.median():.3f}, <0.1 : {100*(vs<0.1).mean():.1f}%")
vd = prereq_kge['dir_score'].dropna()
print(f"Directionnel : >0 (cohérent) {100*(vd>0).mean():.1f}%, <0 {100*(vd<0).mean():.1f}%, =0 {100*(vd==0).mean():.1f}%")

Calcul du critère sémantique...
  Similarité calculée pour 72,343 relations

Chargement concept-course...
  Concepts avec cours : 216,096
Calcul du critère directionnel...
  Score directionnel calculé pour 72,361 relations

=== Distributions ===
Sémantique : mean=0.165, médiane=0.147, <0.1 : 31.5%
Directionnel : >0 (cohérent) 37.1%, <0 36.9%, =0 25.9%


Cellule 4 — le vote majoritaire

In [5]:
import numpy as np

# ===== VOTE MAJORITAIRE (2 critères sur 3) — mêmes seuils qu'hier =====
SEM_THRESH = 0.15
DIR_THRESH = 0.0

prereq_kge['vote_temporal']  = True
prereq_kge['vote_semantic']  = prereq_kge['sem_sim'].fillna(0) >= SEM_THRESH
prereq_kge['vote_direction'] = prereq_kge['dir_score'].fillna(-1) > DIR_THRESH

prereq_kge['n_votes'] = (prereq_kge['vote_temporal'].astype(int) +
                         prereq_kge['vote_semantic'].astype(int) +
                         prereq_kge['vote_direction'].astype(int))

print("Distribution des votes :")
for v in [1, 2, 3]:
    n = (prereq_kge['n_votes'] == v).sum()
    print(f"  {v} vote(s) : {n:,} ({100*n/len(prereq_kge):.1f}%)")

# Garder si >= 2 votes
prereq_kge['keep'] = prereq_kge['n_votes'] >= 2
n_keep = prereq_kge['keep'].sum()
print(f"\n=== Résultat du filtrage (KG KGE, 72 361 relations) ===")
print(f"  Conservées : {n_keep:,} ({100*n_keep/len(prereq_kge):.1f}%)")
print(f"  Éliminées  : {(~prereq_kge['keep']).sum():,} ({100*(~prereq_kge['keep']).mean():.1f}%)")

# Validation : similarité conservées vs éliminées
kept = prereq_kge[prereq_kge['keep']]
removed = prereq_kge[~prereq_kge['keep']]
print(f"\n  Similarité moyenne — conservées : {kept['sem_sim'].mean():.3f}")
print(f"  Similarité moyenne — éliminées  : {removed['sem_sim'].mean():.3f}")

# Sauvegarder les prérequis enrichis (version KGE 72K)
out_dir = r'D:\Memoire\LLMKGER\prereq_enriched'
prereq_final = prereq_kge[prereq_kge['keep']][['head','tail']].copy()
prereq_final['relation'] = 'prerequisite_of'
prereq_final = prereq_final[['head','relation','tail']]
prereq_final.to_csv(os.path.join(out_dir, 'prerequisite_of_kge_enriched.csv'), sep='\t', index=False)
print(f"\n💾 Sauvegardé : prerequisite_of_kge_enriched.csv ({len(prereq_final):,} relations)")

Distribution des votes :
  1 vote(s) : 23,331 (32.2%)
  2 vote(s) : 35,806 (49.5%)
  3 vote(s) : 13,224 (18.3%)

=== Résultat du filtrage (KG KGE, 72 361 relations) ===
  Conservées : 49,030 (67.8%)
  Éliminées  : 23,331 (32.2%)

  Similarité moyenne — conservées : 0.208
  Similarité moyenne — éliminées  : 0.074

💾 Sauvegardé : prerequisite_of_kge_enriched.csv (49,030 relations)


Cellule 5  — Régénérer le KG complet avec prérequis filtrés

In [6]:
import os, time

KG_SAMPLED = r'D:\Memoire\LLMKGER\kg_sampled'
src = os.path.join(KG_SAMPLED, 'triplets_train_kg.tsv')
dst = os.path.join(KG_SAMPLED, 'triplets_train_kg_enriched.tsv')

# Ensemble des prérequis à GARDER (les 49 030 filtrés)
keep_prereq = set(zip(prereq_final['head'].astype(str), prereq_final['tail'].astype(str)))
print(f"Prérequis à garder : {len(keep_prereq):,}")

print("Régénération du KG (remplacement des prérequis)...")
t0 = time.time()
n_total = 0
n_written = 0
n_prereq_kept = 0
n_prereq_removed = 0
n_other = 0

with open(src, 'r', encoding='utf-8') as fin, open(dst, 'w', encoding='utf-8') as fout:
    header = fin.readline()
    fout.write(header)  # garder l'en-tête
    for line in fin:
        parts = line.rstrip().split('\t')
        if len(parts) < 3:
            continue
        h, r, t = parts[0], parts[1], parts[2]
        n_total += 1
        if r == 'prerequisite_of':
            # garder seulement si dans les filtrés
            if (h, t) in keep_prereq:
                fout.write(line)
                n_written += 1
                n_prereq_kept += 1
            else:
                n_prereq_removed += 1
        else:
            # garder toutes les autres relations
            fout.write(line)
            n_written += 1
            n_other += 1

print(f"✅ KG régénéré en {(time.time()-t0):.1f}s")
print(f"\n=== Bilan ===")
print(f"  Triplets originaux : {n_total:,}")
print(f"  Triplets conservés : {n_written:,}")
print(f"  Prérequis gardés   : {n_prereq_kept:,}")
print(f"  Prérequis retirés  : {n_prereq_removed:,}")
print(f"  Autres relations   : {n_other:,}")
print(f"\n💾 Nouveau KG : {dst}")
print(f"   Taille : {os.path.getsize(dst)/1e6:.1f} Mo")

Prérequis à garder : 49,030
Régénération du KG (remplacement des prérequis)...
✅ KG régénéré en 10.1s

=== Bilan ===
  Triplets originaux : 7,915,028
  Triplets conservés : 7,891,697
  Prérequis gardés   : 49,030
  Prérequis retirés  : 23,331
  Autres relations   : 7,842,667

💾 Nouveau KG : D:\Memoire\LLMKGER\kg_sampled\triplets_train_kg_enriched.tsv
   Taille : 268.1 Mo


In [6]:
import json, os

# Lire la config des modèles déjà entraînés (pour reproduire exactement)
KGE_DIR = r'D:\Memoire\LLMKGER\kge_results'

print("=== Configuration des modèles KGE originaux ===\n")
for model_name in ['RotatE', 'TransE', 'ComplEx', 'DistMult', 'TransR']:
    meta_path = os.path.join(KGE_DIR, model_name, 'metadata.json')
    res_path = os.path.join(KGE_DIR, model_name, 'results.json')
    if os.path.exists(meta_path):
        with open(meta_path) as f:
            meta = json.load(f)
        print(f"📊 {model_name}")
        # Afficher les hyperparamètres clés
        print(f"   metadata keys: {list(meta.keys())[:10]}")
        # Résultats
        if os.path.exists(res_path):
            with open(res_path) as f:
                res = json.load(f)
            print(f"   results keys: {list(res.keys())[:5]}")
        print()

# Lire la comparaison originale pour référence
comp_path = os.path.join(KGE_DIR, 'kge_comparison.csv')
if os.path.exists(comp_path):
    import pandas as pd
    comp = pd.read_csv(comp_path)
    print("=== Comparaison KGE originale ===")
    print(comp.to_string())

=== Configuration des modèles KGE originaux ===

📊 RotatE
   metadata keys: []
   results keys: ['losses', 'metrics', 'times']

📊 TransE
   metadata keys: []
   results keys: ['losses', 'metrics', 'times']

📊 ComplEx
   metadata keys: []
   results keys: ['losses', 'metrics', 'times']

📊 DistMult
   metadata keys: []
   results keys: ['losses', 'metrics', 'times']

📊 TransR
   metadata keys: []
   results keys: ['losses', 'metrics', 'times']

=== Comparaison KGE originale ===
  Unnamed: 0     MRR   Hit@1   Hit@3  Hit@10
0     RotatE  0.1975  0.1350  0.2097  0.3167
1     TransE  0.0734  0.0147  0.0920  0.1768
2    ComplEx  0.0205  0.0102  0.0204  0.0396
3   DistMult  0.0007  0.0004  0.0006  0.0009


In [8]:
import json, os

KGE_DIR = r'D:\Memoire\LLMKGER\kge_results'

# Lire les métriques de TOUS les modèles, y compris TransR
print("=== Métriques de tous les modèles (dont TransR) ===\n")
for model_name in ['RotatE', 'TransE', 'ComplEx', 'DistMult', 'TransR']:
    res_path = os.path.join(KGE_DIR, model_name, 'results.json')
    if os.path.exists(res_path):
        with open(res_path) as f:
            res = json.load(f)
        metrics = res.get('metrics', {})
        # Chercher MRR et Hit@k dans la structure
        print(f"📊 {model_name}:")
        # PyKEEN stocke les métriques dans une structure imbriquée
        if 'both' in metrics:
            realistic = metrics['both'].get('realistic', {})
            mrr = realistic.get('inverse_harmonic_mean_rank', 'N/A')
            h1 = realistic.get('hits_at_1', 'N/A')
            h10 = realistic.get('hits_at_10', 'N/A')
            print(f"   MRR={mrr}, Hit@1={h1}, Hit@10={h10}")
        else:
            print(f"   Structure: {list(metrics.keys())[:5]}")
        # Temps d'entraînement
        times = res.get('times', {})
        print(f"   Temps: {times}")
        print()

=== Métriques de tous les modèles (dont TransR) ===

📊 RotatE:
   MRR=0.19746966660022736, Hit@1=0.13499701811750672, Hit@10=0.3167478972587246
   Temps: {'evaluation': 3220.01584649086, 'training': 2278.9816648960114}

📊 TransE:
   MRR=0.07340438663959503, Hit@1=0.014703765397824253, Hit@10=0.17676393773006766
   Temps: {'evaluation': 2282.366415500641, 'training': 991.4963526725769}

📊 ComplEx:
   MRR=0.02048804983496666, Hit@1=0.010230941657926667, Hit@10=0.039597343039874965
   Temps: {'evaluation': 1856.342526435852, 'training': 1797.2138268947601}

📊 DistMult:
   MRR=0.0006555500440299511, Hit@1=0.0004421411972772328, Hit@10=0.0008740000411294137
   Temps: {'evaluation': 766.3273646831512, 'training': 946.51225066185}

📊 TransR:
   MRR=0.01244588103145361, Hit@1=0.003166964854915993, Hit@10=0.02626113064758262
   Temps: {'evaluation': 749.7711882591248, 'training': 1977.2208337783813}



In [10]:
import json

nb_path = r'D:\Memoire\LLMKGER\notebooks\04_kge_training.ipynb'
with open(nb_path, 'r', encoding='utf-8') as f:
    nb = json.load(f)

print("=== Cellules contenant 'pipeline' ou 'model' (config KGE) ===\n")
for i, cell in enumerate(nb['cells']):
    if cell['cell_type'] == 'code':
        src = ''.join(cell['source'])
        # Chercher les cellules d'entraînement
        if 'pipeline(' in src or 'embedding_dim' in src or "model='" in src or 'model="' in src:
            print(f"--- Cellule {i} ---")
            print(src[:1500])
            print()

=== Cellules contenant 'pipeline' ou 'model' (config KGE) ===

--- Cellule 6 ---
import torch
from pykeen.pipeline import pipeline
import json, os

RESULTS_DIR = r'D:\Memoire\LLMKGER\kge_results'
os.makedirs(RESULTS_DIR, exist_ok=True)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

# Configuration commune
COMMON_CONFIG = {
    'num_epochs'      : 100,
    'embedding_dim'   : 128,
    'batch_size'      : 1024,
    'device'          : DEVICE,
    'random_seed'     : 42,
}

# Configurations specifiques par modele
MODELS_CONFIG = {
    'TransE': {
        'model': 'TransE',
        'model_kwargs': {'embedding_dim': COMMON_CONFIG['embedding_dim'],
                         'scoring_fct_norm': 1},
        'loss': 'MarginRankingLoss',
        'loss_kwargs': {'margin': 1.0},
        'optimizer': 'Adam',
        'optimizer_kwargs': {'lr': 0.001},
    },
    'RotatE': {
        'model': 'RotatE',
        'model_kwargs': {'embedding_dim': COMMON_CONFIG['embedding_dim']},
        'loss':

In [11]:
import json

nb_path = r'D:\Memoire\LLMKGER\notebooks\04_kge_training.ipynb'
with open(nb_path, 'r', encoding='utf-8') as f:
    nb = json.load(f)

print("=== Cellules de préparation des données (TriplesFactory) ===\n")
for i, cell in enumerate(nb['cells']):
    if cell['cell_type'] == 'code':
        src = ''.join(cell['source'])
        if 'TriplesFactory' in src or 'train_tf' in src or 'from_path' in src or 'from_labeled' in src:
            print(f"--- Cellule {i} ---")
            print(src[:1800])
            print()

=== Cellules de préparation des données (TriplesFactory) ===

--- Cellule 3 ---
import os
import pandas as pd
from pykeen.triples import TriplesFactory

KG_DIR = r'D:\Memoire\LLMKGER\kg_final'

# Charger kg_train.tsv et EXCLURE les follows
df = pd.read_csv(os.path.join(KG_DIR, 'kg_train.tsv'),
                 sep='\t', header=0, names=['head','relation','tail'])

# Garder seulement les triplets KG (hors follows)
df_kg = df[df['relation'] != 'follows'].copy()

print(f'Triplets totaux    : {len(df):,}')
print(f'Triplets KG only   : {len(df_kg):,}')
print(f'Relations distinctes : {df_kg["relation"].nunique()}')
print()
print(df_kg['relation'].value_counts())

--- Cellule 4 ---
tf = TriplesFactory.from_labeled_triples(
    triples=df_kg[['head','relation','tail']].values,
    create_inverse_triples=False
)

train_tf, valid_tf, test_tf = tf.split([0.85, 0.075, 0.075], random_state=42)

print(f'Train triplets : {train_tf.num_triples:,}')
print(f'Valid triplets : {valid_tf.num_triples:,}')
p

In [12]:
import pandas as pd, os

KG_FINAL = r'D:\Memoire\LLMKGER\kg_final'
kg_train = os.path.join(KG_FINAL, 'kg_train.tsv')

print("=== kg_final/kg_train.tsv (source de la comparaison KGE) ===\n")
print(f"Taille : {os.path.getsize(kg_train)/1e6:.1f} Mo")

# Aperçu
with open(kg_train, 'r', encoding='utf-8') as f:
    for i, line in enumerate(f):
        if i >= 3: break
        print(f"  {line.rstrip()}")

# Distribution des relations
print("\nDistribution des relations :")
rel_counts = {}
with open(kg_train, 'r', encoding='utf-8') as f:
    header = f.readline()  # skip header
    for line in f:
        parts = line.rstrip().split('\t')
        if len(parts) >= 3:
            rel_counts[parts[1]] = rel_counts.get(parts[1], 0) + 1
for rel, cnt in sorted(rel_counts.items(), key=lambda x: -x[1]):
    print(f"  {rel}: {cnt:,}")

=== kg_final/kg_train.tsv (source de la comparaison KGE) ===

Taille : 188.9 Mo
  head	relation	tail
  U_23968068	watches	9845A467C89E9D009C33DC5901307461
  U_31017024	watches	9B15701E05D259349C33DC5901307461

Distribution des relations :
  follows: 5,044,890
  has_concept: 251,334
  belongs_to: 211,691
  watches: 112,038
  prerequisite_of: 59,751
  taught_by: 10,742
  offered_by: 2,793


In [13]:
import pandas as pd, os, time

KG_FINAL = r'D:\Memoire\LLMKGER\kg_final'
PREREQ_DIR = r'D:\Memoire\LLMKGER\prereq_enriched'

src = os.path.join(KG_FINAL, 'kg_train.tsv')
dst = os.path.join(KG_FINAL, 'kg_train_enriched.tsv')

# Charger les 41 272 prérequis enrichis (ceux d'hier, pour le KG RecBole/final)
prereq_enr = pd.read_csv(os.path.join(PREREQ_DIR, 'prerequisite_of_enriched.csv'), sep='\t')
keep_prereq = set(zip(prereq_enr['head'].astype(str), prereq_enr['tail'].astype(str)))
print(f"Prérequis enrichis à garder : {len(keep_prereq):,}")

print("Régénération de kg_final/kg_train_enriched.tsv...")
t0 = time.time()
n_kept_prereq = 0; n_removed_prereq = 0; n_other = 0

with open(src, 'r', encoding='utf-8') as fin, open(dst, 'w', encoding='utf-8') as fout:
    header = fin.readline()
    fout.write(header)
    for line in fin:
        parts = line.rstrip().split('\t')
        if len(parts) < 3: continue
        h, r, t = parts[0], parts[1], parts[2]
        if r == 'prerequisite_of':
            if (h, t) in keep_prereq:
                fout.write(line); n_kept_prereq += 1
            else:
                n_removed_prereq += 1
        else:
            fout.write(line); n_other += 1

print(f"✅ Régénéré en {time.time()-t0:.1f}s")
print(f"  Prérequis gardés  : {n_kept_prereq:,}")
print(f"  Prérequis retirés : {n_removed_prereq:,}")
print(f"  Autres relations  : {n_other:,}")
print(f"\n💾 {dst} ({os.path.getsize(dst)/1e6:.1f} Mo)")

Prérequis enrichis à garder : 41,272
Régénération de kg_final/kg_train_enriched.tsv...
✅ Régénéré en 9.5s
  Prérequis gardés  : 41,272
  Prérequis retirés : 18,479
  Autres relations  : 5,633,488

💾 D:\Memoire\LLMKGER\kg_final\kg_train_enriched.tsv (187.3 Mo)


la comparaison des 5 modèles KGE

In [7]:
import os, torch, json, time
import pandas as pd
from pykeen.triples import TriplesFactory
from pykeen.pipeline import pipeline

KG_FINAL = r'D:\Memoire\LLMKGER\kg_final'
RESULTS_DIR = r'D:\Memoire\LLMKGER\kge_results_enriched'  # NOUVEAU dossier
os.makedirs(RESULTS_DIR, exist_ok=True)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

# 1. Charger le KG enrichi et EXCLURE les follows (comme l'original)
print("Chargement du KG enrichi...")
df = pd.read_csv(os.path.join(KG_FINAL, 'kg_train_enriched.tsv'),
                 sep='\t', header=0, names=['head','relation','tail'])
df_kg = df[df['relation'] != 'follows'].copy()
print(f"Triplets KG (hors follows) : {len(df_kg):,}")
print(df_kg['relation'].value_counts())

# 2. Créer les TriplesFactory (MÊME split, MÊME seed que l'original)
tf = TriplesFactory.from_labeled_triples(
    triples=df_kg[['head','relation','tail']].values,
    create_inverse_triples=False
)
train_tf, valid_tf, test_tf = tf.split([0.85, 0.075, 0.075], random_state=42)
print(f"\nTrain:{train_tf.num_triples:,} Valid:{valid_tf.num_triples:,} Test:{test_tf.num_triples:,}")
print(f"Entités:{tf.num_entities:,} Relations:{tf.num_relations:,}")

Chargement du KG enrichi...
Triplets KG (hors follows) : 629,870
relation
has_concept        251334
belongs_to         211691
watches            112038
prerequisite_of     41272
taught_by           10742
offered_by           2793
Name: count, dtype: int64

Train:535,389 Valid:47,240 Test:47,241
Entités:141,591 Relations:6


In [8]:
# Vérifier si des entités du test sont absentes du train
train_entities = set(train_tf.entity_to_id.keys())
test_entities = set()
import numpy as np

# Entités dans le test
test_triples = test_tf.triples
for h, r, t in test_triples:
    test_entities.add(h)
    test_entities.add(t)

# Entités du test absentes du train
orphan = test_entities - train_entities
print(f"Entités dans le test : {len(test_entities):,}")
print(f"Entités dans le train : {len(train_entities):,}")
print(f"Entités du test ABSENTES du train : {len(orphan):,}")

if len(orphan) > 0:
    print(f"\n⚠️ PROBLÈME CONFIRMÉ : {len(orphan)} entités orphelines")
    print("   → C'est la cause du MRR=0")
    print("   → Solution : refaire le split avec filtrage des entités")
else:
    print("\n✅ Pas d'entités orphelines — le problème est ailleurs")

Reconstructing all label-based triples. This is expensive and rarely needed.


Entités dans le test : 39,529
Entités dans le train : 141,591
Entités du test ABSENTES du train : 0

✅ Pas d'entités orphelines — le problème est ailleurs


In [18]:
import pykeen, torch, numpy
print(f"PyKEEN  : {pykeen.get_version()}")
print(f"PyTorch : {torch.__version__}")
print(f"NumPy   : {numpy.__version__}")

# La doc originale disait PyKEEN 1.11.1 — vérifions si c'est toujours le cas

PyKEEN  : 1.11.1
PyTorch : 2.6.0+cu124
NumPy   : 1.26.4


In [9]:
import os, torch, time
import pandas as pd
from pykeen.models import TransE, RotatE, ComplEx, DistMult
from pykeen.training import SLCWATrainingLoop
from pykeen.losses import MarginRankingLoss, NSSALoss, SoftplusLoss
from pykeen.triples import TriplesFactory

RESULTS_DIR = r'D:\Memoire\LLMKGER\kge_results_enriched'
os.makedirs(RESULTS_DIR, exist_ok=True)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

def train_model(model_name, model_class, loss_obj, optimizer_cls, lr, model_extra=None):
    print(f'\n{"="*50}\n  Training : {model_name}\n{"="*50}')
    t0 = time.time()
    torch.manual_seed(42)

    kwargs = {'triples_factory': train_tf, 'loss': loss_obj}
    if model_extra:
        kwargs.update(model_extra)
    model = model_class(**kwargs).to(DEVICE)

    optimizer = optimizer_cls(model.get_grad_params(), lr=lr)

    training_loop = SLCWATrainingLoop(
        model=model, triples_factory=train_tf, optimizer=optimizer,
    )
    losses = training_loop.train(
        triples_factory=train_tf, num_epochs=100, batch_size=1024,
        use_tqdm=True,
    )
    # Sauvegarder SEULEMENT le modèle (pas d'évaluation)
    mdir = os.path.join(RESULTS_DIR, model_name)
    os.makedirs(mdir, exist_ok=True)
    torch.save(model, os.path.join(mdir, 'trained_model.pkl'))
    # Sauvegarder aussi la TriplesFactory pour l'évaluation ultérieure
    train_tf.to_path_binary(os.path.join(mdir, 'training_triples'))
    print(f'  ✅ {model_name} entraîné — {(time.time()-t0)/60:.1f} min')
    print(f'     Losses (5 derniers) : {[round(l,4) for l in losses[-5:]]}')
    return losses

# Entraîner les 4 modèles (SANS évaluation)
train_model('TransE', TransE,
            MarginRankingLoss(margin=1.0), torch.optim.Adam, 0.001,
            {'embedding_dim':128, 'scoring_fct_norm':1})

train_model('RotatE', RotatE,
            NSSALoss(), torch.optim.Adam, 0.001,
            {'embedding_dim':128})

train_model('ComplEx', ComplEx,
            SoftplusLoss(), torch.optim.Adagrad, 0.1,
            {'embedding_dim':128})

train_model('DistMult', DistMult,
            SoftplusLoss(), torch.optim.Adagrad, 0.1,
            {'embedding_dim':128})

print("\n✅ Training des 4 modèles terminé (sans évaluation)")

No random seed is specified. This may lead to non-reproducible results.



  Training : TransE


Training epochs on cuda:0:   0%|          | 0/100 [00:00<?, ?epoch/s]

Training batches on cuda:0:   0%|          | 0.00/523 [00:00<?, ?batch/s]

Training batches on cuda:0:   0%|          | 0.00/523 [00:00<?, ?batch/s]

Training batches on cuda:0:   0%|          | 0.00/523 [00:00<?, ?batch/s]

... (403 outputs left out)

Training batches on cuda:0:   0%|          | 0.00/523 [00:00<?, ?batch/s]

Training batches on cuda:0:   0%|          | 0.00/523 [00:00<?, ?batch/s]

Training batches on cuda:0:   0%|          | 0.00/523 [00:00<?, ?batch/s]

  ✅ DistMult entraîné — 21.0 min
     Losses (5 derniers) : [0.6893, 0.6893, 0.6893, 0.6893, 0.6893]

✅ Training des 4 modèles terminé (sans évaluation)


In [10]:
import os, torch, json
from pykeen.evaluation import RankBasedEvaluator
import numpy as np

RESULTS_DIR = r'D:\Memoire\LLMKGER\kge_results_enriched'
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

all_results = {}

for model_name in ['TransE', 'RotatE', 'ComplEx', 'DistMult']:
    print(f'\n{"="*50}\n  Évaluation : {model_name}\n{"="*50}')
    try:
        # Recharger le modèle
        mdir = os.path.join(RESULTS_DIR, model_name)
        model = torch.load(os.path.join(mdir, 'trained_model.pkl'), weights_only=False)
        model = model.to(DEVICE)
        model.eval()

        # Évaluateur filtré
        evaluator = RankBasedEvaluator(filtered=True)

        # Évaluer sur le test (avec batch_size réduit pour éviter l'overflow)
        results = evaluator.evaluate(
            model=model,
            mapped_triples=test_tf.mapped_triples,
            additional_filter_triples=[train_tf.mapped_triples, valid_tf.mapped_triples],
            batch_size=16,   # petit batch pour éviter l'overflow
            use_tqdm=True,
        )

        m = results.to_dict()
        mrr = m['both']['realistic']['inverse_harmonic_mean_rank']
        h1  = m['both']['realistic']['hits_at_1']
        h3  = m['both']['realistic']['hits_at_3']
        h10 = m['both']['realistic']['hits_at_10']

        all_results[model_name] = {'MRR':round(mrr,4),'Hit@1':round(h1,4),
                                    'Hit@3':round(h3,4),'Hit@10':round(h10,4)}
        print(f'  MRR:{mrr:.4f} Hit@1:{h1:.4f} Hit@3:{h3:.4f} Hit@10:{h10:.4f}')
    except Exception as e:
        print(f'  ❌ Erreur : {e}')
        all_results[model_name] = None

print("\n✅ Évaluation terminée")
print(all_results)


  Évaluation : TransE


Evaluating on cuda:0:   0%|          | 0.00/47.2k [00:00<?, ?triple/s]

  MRR:0.0734 Hit@1:0.0153 Hit@3:0.0929 Hit@10:0.1755

  Évaluation : RotatE


Evaluating on cuda:0:   0%|          | 0.00/47.2k [00:00<?, ?triple/s]

  MRR:0.2012 Hit@1:0.1406 Hit@3:0.2114 Hit@10:0.3174

  Évaluation : ComplEx


Evaluating on cuda:0:   0%|          | 0.00/47.2k [00:00<?, ?triple/s]

  MRR:0.0171 Hit@1:0.0079 Hit@3:0.0166 Hit@10:0.0334

  Évaluation : DistMult


Evaluating on cuda:0:   0%|          | 0.00/47.2k [00:00<?, ?triple/s]

  MRR:0.0007 Hit@1:0.0004 Hit@3:0.0007 Hit@10:0.0010

✅ Évaluation terminée
{'TransE': {'MRR': 0.0734, 'Hit@1': 0.0153, 'Hit@3': 0.0929, 'Hit@10': 0.1755}, 'RotatE': {'MRR': 0.2012, 'Hit@1': 0.1406, 'Hit@3': 0.2114, 'Hit@10': 0.3174}, 'ComplEx': {'MRR': 0.0171, 'Hit@1': 0.0079, 'Hit@3': 0.0166, 'Hit@10': 0.0334}, 'DistMult': {'MRR': 0.0007, 'Hit@1': 0.0004, 'Hit@3': 0.0007, 'Hit@10': 0.001}}


In [11]:
import pandas as pd, os

RESULTS_DIR = r'D:\Memoire\LLMKGER\kge_results_enriched'

df_results = pd.DataFrame(all_results).T
df_results = df_results.sort_values('MRR', ascending=False)

print('='*55)
print('  Comparaison KGE — KG ENRICHI (prérequis multi-critères)')
print('='*55)
print(df_results.to_string())
print('='*55)

best = df_results.index[0]
print(f'\n🏆 Meilleur modèle : {best} (MRR={df_results.loc[best,"MRR"]})')

# Comparaison avec l'original
print(f'\n=== Comparaison original vs enrichi ===')
orig = {'RotatE':0.1975, 'TransE':0.0734, 'ComplEx':0.0205, 'DistMult':0.0007}
for m in df_results.index:
    o = orig.get(m, 0)
    n = df_results.loc[m,'MRR']
    print(f'  {m:10s} : {o:.4f} → {n:.4f} ({n-o:+.4f})')

df_results.to_csv(os.path.join(RESULTS_DIR, 'kge_comparison_enriched.csv'))
print(f'\n💾 Tableau sauvegardé : kge_comparison_enriched.csv')

  Comparaison KGE — KG ENRICHI (prérequis multi-critères)
             MRR   Hit@1   Hit@3  Hit@10
RotatE    0.2012  0.1406  0.2114  0.3174
TransE    0.0734  0.0153  0.0929  0.1755
ComplEx   0.0171  0.0079  0.0166  0.0334
DistMult  0.0007  0.0004  0.0007  0.0010

🏆 Meilleur modèle : RotatE (MRR=0.2012)

=== Comparaison original vs enrichi ===
  RotatE     : 0.1975 → 0.2012 (+0.0037)
  TransE     : 0.0734 → 0.0734 (+0.0000)
  ComplEx    : 0.0205 → 0.0171 (-0.0034)
  DistMult   : 0.0007 → 0.0007 (+0.0000)

💾 Tableau sauvegardé : kge_comparison_enriched.csv


In [12]:
import json

nb_path = r'D:\Memoire\LLMKGER\notebooks\04b_rotate_v2_with_follows.ipynb'
with open(nb_path, 'r', encoding='utf-8') as f:
    nb = json.load(f)

print("=== Cellules de RotatE v2 (avec follows) ===\n")
for i, cell in enumerate(nb['cells']):
    if cell['cell_type'] == 'code':
        src = ''.join(cell['source'])
        # Chercher les cellules clés
        if any(kw in src for kw in ['pipeline(', 'RotatE', 'follows', 'TriplesFactory', 'from_labeled', 'kg_train']):
            print(f"--- Cellule {i} ---")
            print(src[:1500])
            print()

=== Cellules de RotatE v2 (avec follows) ===

--- Cellule 3 ---
SAMPLE_FOLLOWS = 500_000  # sous-échantillon de follows

# ── 1. Charger triplets KG sémantique (hors follows) ──────────────────────
df_kg = pd.read_csv(
    os.path.join(KG_DIR, 'kg_train.tsv'),
    sep='\t', header=0, names=['head','relation','tail']
)
df_kg_sem = df_kg[df_kg['relation'] != 'follows'].copy()
print(f'Triplets KG sémantique : {len(df_kg_sem):,}')

# ── 2. Charger interactions follows (train seulement) ─────────────────────
df_follows = pd.read_csv(
    os.path.join(KG_DIR, 'interactions_train.tsv'),
    sep='\t', header=0, names=['head','relation','tail']
)
print(f'Follows train total    : {len(df_follows):,}')

# ── 3. Sous-échantillon de follows ────────────────────────────────────────
df_follows_sample = df_follows.sample(n=SAMPLE_FOLLOWS, random_state=42)
print(f'Follows sous-échantillon : {len(df_follows_sample):,}')

# ── 4. Combiner KG sémantique + follows ───────────────────────────────────
df_com

In [1]:
import os, torch, time
import pandas as pd
from pykeen.triples import TriplesFactory
from pykeen.models import RotatE
from pykeen.training import SLCWATrainingLoop
from pykeen.losses import NSSALoss

# Patch torch.load
_orig = torch.load
torch.load = lambda *a, **k: _orig(*a, **{**k, 'weights_only': False})

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"✅ Imports OK, Device: {DEVICE}")

✅ Imports OK, Device: cuda


In [2]:
import os, torch, time
import pandas as pd
from pykeen.triples import TriplesFactory
from pykeen.models import RotatE
from pykeen.training import SLCWATrainingLoop
from pykeen.losses import NSSALoss

KG_DIR = r'D:\Memoire\LLMKGER\kg_final'
RESULTS_DIR = r'D:\Memoire\LLMKGER\kge_results_v2_enriched'  # NOUVEAU
os.makedirs(RESULTS_DIR, exist_ok=True)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
SAMPLE_FOLLOWS = 500_000

# 1. KG sémantique ENRICHI (hors follows)
print("Chargement du KG enrichi (sémantique)...")
df_kg = pd.read_csv(os.path.join(KG_DIR, 'kg_train_enriched.tsv'),
                    sep='\t', header=0, names=['head','relation','tail'])
df_kg_sem = df_kg[df_kg['relation'] != 'follows'].copy()
print(f"  Triplets KG sémantique enrichi : {len(df_kg_sem):,}")
print(f"  Dont prerequisite_of : {(df_kg_sem['relation']=='prerequisite_of').sum():,}")

# 2. Follows (train)
df_follows = pd.read_csv(os.path.join(KG_DIR, 'interactions_train.tsv'),
                         sep='\t', header=0, names=['head','relation','tail'])
print(f"  Follows train total : {len(df_follows):,}")

# 3. Sous-échantillon follows (MÊME seed que l'original)
df_follows_sample = df_follows.sample(n=SAMPLE_FOLLOWS, random_state=42)
print(f"  Follows sous-échantillon : {len(df_follows_sample):,}")

# 4. Combiner
df_combined = pd.concat([df_kg_sem, df_follows_sample], ignore_index=True)
print(f"  Dataset combiné : {len(df_combined):,}")

# 5. TriplesFactory + split (MÊME seed)
tf = TriplesFactory.from_labeled_triples(
    triples=df_combined[['head','relation','tail']].values,
    create_inverse_triples=False)
train_tf_v2, valid_tf_v2, test_tf_v2 = tf.split([0.85, 0.075, 0.075], random_state=42)
print(f"  Train:{train_tf_v2.num_triples:,} Entités:{tf.num_entities:,} Relations:{tf.num_relations:,}")
print(f"  Relations : {list(tf.relation_to_id.keys())}")

# 6. Entraîner RotatE v2 (training PUR, sans évaluation → pas d'overflow)
print("\nEntraînement RotatE v2 enrichi...")
t0 = time.time()
torch.manual_seed(42)
model_v2 = RotatE(triples_factory=train_tf_v2, embedding_dim=128, loss=NSSALoss()).to(DEVICE)
optimizer = torch.optim.Adam(model_v2.get_grad_params(), lr=0.001)
training_loop = SLCWATrainingLoop(model=model_v2, triples_factory=train_tf_v2, optimizer=optimizer)
losses = training_loop.train(triples_factory=train_tf_v2, num_epochs=100, batch_size=1024, use_tqdm=True)

# 7. Sauvegarder le modèle + les mappings
mdir = os.path.join(RESULTS_DIR, 'RotatE_v2')
os.makedirs(mdir, exist_ok=True)
torch.save(model_v2, os.path.join(mdir, 'trained_model.pkl'))
train_tf_v2.to_path_binary(os.path.join(mdir, 'training_triples'))
print(f"\n✅ RotatE v2 enrichi entraîné — {(time.time()-t0)/60:.1f} min")
print(f"   Losses (5 derniers) : {[round(l,4) for l in losses[-5:]]}")
print(f"   Sauvegardé : {mdir}")

Chargement du KG enrichi (sémantique)...
  Triplets KG sémantique enrichi : 629,870
  Dont prerequisite_of : 41,272
  Follows train total : 5,044,890
  Follows sous-échantillon : 500,000
  Dataset combiné : 1,129,870


No random seed is specified. This may lead to non-reproducible results.


  Train:960,389 Entités:235,055 Relations:7
  Relations : ['belongs_to', 'follows', 'has_concept', 'offered_by', 'prerequisite_of', 'taught_by', 'watches']

Entraînement RotatE v2 enrichi...


Training epochs on cuda:0:   0%|          | 0/100 [00:00<?, ?epoch/s]

Training batches on cuda:0:   0%|          | 0.00/938 [00:00<?, ?batch/s]

Training batches on cuda:0:   0%|          | 0.00/938 [00:00<?, ?batch/s]

... (95 outputs left out)

Training batches on cuda:0:   0%|          | 0.00/938 [00:00<?, ?batch/s]

Training batches on cuda:0:   0%|          | 0.00/938 [00:00<?, ?batch/s]

Training batches on cuda:0:   0%|          | 0.00/938 [00:00<?, ?batch/s]


✅ RotatE v2 enrichi entraîné — 78.1 min
   Losses (5 derniers) : [0.0609, 0.0611, 0.0608, 0.0606, 0.0606]
   Sauvegardé : D:\Memoire\LLMKGER\kge_results_v2_enriched\RotatE_v2


In [3]:
import os

mdir = r'D:\Memoire\LLMKGER\kge_results_v2_enriched\RotatE_v2'
training_path = os.path.join(mdir, 'training_triples')

print("=== Contenu du dossier training_triples ===")
if os.path.exists(training_path):
    for f in os.listdir(training_path):
        fp = os.path.join(training_path, f)
        print(f"  {f} ({os.path.getsize(fp)/1024:.1f} Ko)")
else:
    print("  training_triples absent")
    print(f"\n  Contenu de {mdir} :")
    for f in os.listdir(mdir):
        print(f"    {f}")

=== Contenu du dossier training_triples ===
  base.pth (0.9 Ko)
  entity_to_id.tsv.gz (1800.5 Ko)
  numeric_triples.tsv.gz (5136.8 Ko)
  relation_to_id.tsv.gz (0.1 Ko)


In [4]:
import os, torch, gzip
import pandas as pd
import numpy as np
from tqdm import tqdm

RESULTS_V2 = r'D:\Memoire\LLMKGER\kge_results_v2_enriched\RotatE_v2'
RESULTS_REC = r'D:\Memoire\LLMKGER\reco_results_final'
DEVICE = 'cpu'  # CPU pour éviter les soucis mémoire GPU

# 1. Charger le modèle RotatE v2 enrichi
print("Chargement de RotatE v2 enrichi...")
model_v2 = torch.load(os.path.join(RESULTS_V2, 'trained_model.pkl'),
                      map_location='cpu', weights_only=False)
model_v2.eval()

# 2. Charger les mappings
training_path = os.path.join(RESULTS_V2, 'training_triples')
ent_df = pd.read_csv(os.path.join(training_path, 'entity_to_id.tsv.gz'),
                     sep='\t', compression='gzip')
rel_df = pd.read_csv(os.path.join(training_path, 'relation_to_id.tsv.gz'),
                     sep='\t', compression='gzip')
entity2id = dict(zip(ent_df['label'].astype(str), ent_df['id']))
relation2id = dict(zip(rel_df['label'].astype(str), rel_df['id']))
follows_id = relation2id.get('follows', -1)
print(f"  Entités : {len(entity2id):,}")
print(f"  follows_id : {follows_id}")

# 3. Fonction S_KG pour un batch de paires (user, course)
@torch.no_grad()
def compute_skg_batch(user_ids, course_ids):
    scores = np.full(len(user_ids), np.nan, dtype=np.float32)
    valid_h, valid_r, valid_t, valid_pos = [], [], [], []
    for k, (u, c) in enumerate(zip(user_ids, course_ids)):
        if u in entity2id and c in entity2id and follows_id != -1:
            valid_h.append(entity2id[u])
            valid_r.append(follows_id)
            valid_t.append(entity2id[c])
            valid_pos.append(k)
    if valid_h:
        hrt = torch.tensor(list(zip(valid_h, valid_r, valid_t)), dtype=torch.long)
        s = model_v2.score_hrt(hrt).squeeze(-1).cpu().numpy()
        for pos, val in zip(valid_pos, s):
            scores[pos] = val
    return scores

# Test rapide
test_score = compute_skg_batch(['U_12868687'], ['C_934517'])
print(f"  Test S_KG (U_12868687, C_934517) : {test_score[0]}")
print("✅ Fonction S_KG prête")

Chargement de RotatE v2 enrichi...
  Entités : 235,055
  follows_id : 1
  Test S_KG (U_12868687, C_934517) : -7.7781853675842285
✅ Fonction S_KG prête


In [5]:
import pandas as pd, os, numpy as np
from tqdm import tqdm

RESULTS_REC = r'D:\Memoire\LLMKGER\reco_results_final'

# Ordre des users (lignes) et cours candidats (colonnes)
df_ref = pd.read_csv(os.path.join(RESULTS_REC, 'S_emb_scores_final.csv'))
users_order = df_ref['user_id'].astype(str).tolist()
N = len(users_order)

# L'ordre des cours candidats : on le récupère depuis une matrice existante + le mapping
# On a besoin de candidate_courses dans le bon ordre (2812 colonnes)
# Reconstruire depuis le KG RecBole (comme dans les notebooks précédents)
# Le plus sûr : charger l'ordre depuis un fichier sauvegardé

# Vérifier si on a candidate_courses quelque part
print("Recherche de l'ordre des cours candidats...")

# Option : reconstruire depuis le CSV S_KG original (qui a les mêmes colonnes)
# En fait, l'ordre des colonnes = candidate_courses de RecBole
# On doit le régénérer. Vérifions les fichiers disponibles
for f in os.listdir(RESULTS_REC):
    if 'candidate' in f.lower() or 'course' in f.lower() or 'item' in f.lower():
        print(f"  Trouvé : {f}")

# Charger la matrice S_KG existante pour connaître le nombre de colonnes
mat_skg_old = np.load(os.path.join(RESULTS_REC, 'mat_S_KG_norm.npy'), mmap_mode='r')
print(f"\nMatrice S_KG existante : {mat_skg_old.shape}")
print(f"Users : {N}, Cours candidats : {mat_skg_old.shape[1]}")

Recherche de l'ordre des cours candidats...

Matrice S_KG existante : (53897, 2812)
Users : 53897, Cours candidats : 2812


In [6]:
import gc
from recbole.config import Config
from recbole.data import create_dataset, data_preparation
from recbole.utils import init_seed, init_logger

RECBOLE_ROOT = r'D:\Memoire\LLMKGER\recbole_data_large'

base_config = {
    'data_path': RECBOLE_ROOT, 'dataset': 'llmkger',
    'USER_ID_FIELD':'user_id','ITEM_ID_FIELD':'item_id','RATING_FIELD':'rating',
    'load_col':{'inter':['user_id','item_id','rating']},
    'eval_args':{'split':{'LS':'valid_and_test'},'order':'RO','group_by':'user','mode':'full'},
    'metrics':['NDCG'],'topk':[10],'valid_metric':'NDCG@10',
    'epochs':1,'device':'cpu','seed':42,'show_progress':False,
}
config = Config(model='LightGCN', dataset='llmkger', config_dict=base_config)
init_seed(config['seed'], config['reproducibility']); init_logger(config)
dataset = create_dataset(config)

n_items = dataset.num(config['ITEM_ID_FIELD'])
item2id = dict(dataset.field2token_id[config['ITEM_ID_FIELD']])
id2item = {v:k for k,v in item2id.items()}
candidate_items = [i for i in range(n_items) if id2item.get(i,'[PAD]')!='[PAD]']
candidate_courses = [id2item[i] for i in candidate_items]

print(f"Cours candidats : {len(candidate_courses):,}")
print(f"Premiers cours : {candidate_courses[:5]}")

# Libérer la mémoire
del dataset
gc.collect()
print("✅ Ordre des cours candidats récupéré")

c:\Users\AMIRA\anaconda3\envs\llmkger\lib\site-packages\recbole\data\dataset\dataset.py:648: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  feat[field].fillna(value=0, inplace=True)
c:\Users\AMIRA\anaconda3\envs\llmkger\lib\site-packages\recbole\data\dataset\dataset.py:650: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are sett

Cours candidats : 2,812
Premiers cours : ['C_681678', 'C_629559', 'C_696549', 'C_782555', 'C_735405']
✅ Ordre des cours candidats récupéré


In [7]:
import numpy as np, torch
from tqdm import tqdm

print("Calcul de la matrice S_KG enrichie...")
print(f"Users : {N}, Cours : {len(candidate_courses)}")

# Pré-mapper les cours candidats vers leurs IDs RotatE (colonnes)
course_rotate_id = []
for c in candidate_courses:
    course_rotate_id.append(entity2id.get(c, -1))
course_rotate_id = np.array(course_rotate_id)
n_valid_courses = (course_rotate_id >= 0).sum()
print(f"Cours dans RotatE : {n_valid_courses}/{len(candidate_courses)}")

# Calculer S_KG ligne par ligne (user par user)
mat_skg_new = np.full((N, len(candidate_courses)), np.nan, dtype=np.float32)

@torch.no_grad()
def skg_user_all_courses(u_ext):
    """Score S_KG d'un user pour tous les cours candidats."""
    u_id = entity2id.get(u_ext, -1)
    if u_id < 0 or follows_id < 0:
        return None
    # Construire les triplets (u, follows, tous les cours valides)
    valid_mask = course_rotate_id >= 0
    valid_courses = course_rotate_id[valid_mask]
    h = np.full(len(valid_courses), u_id)
    r = np.full(len(valid_courses), follows_id)
    hrt = torch.tensor(np.stack([h, r, valid_courses], axis=1), dtype=torch.long)
    scores = model_v2.score_hrt(hrt).squeeze(-1).cpu().numpy()
    # Remettre dans le vecteur complet
    full = np.full(len(candidate_courses), np.nan, dtype=np.float32)
    full[valid_mask] = scores
    return full

n_found = 0
for row, u_ext in enumerate(tqdm(users_order)):
    res = skg_user_all_courses(u_ext)
    if res is not None:
        mat_skg_new[row] = res
        n_found += 1

print(f"\n✅ Matrice S_KG enrichie calculée")
print(f"   Shape : {mat_skg_new.shape}")
print(f"   Users trouvés dans RotatE : {n_found}/{N}")
print(f"   Valeurs NaN : {np.isnan(mat_skg_new).sum():,}")
print(f"   Moyenne (hors NaN) : {np.nanmean(mat_skg_new):.3f}")

Calcul de la matrice S_KG enrichie...
Users : 53897, Cours : 2812
Cours dans RotatE : 2812/2812


100%|██████████| 53897/53897 [02:35<00:00, 346.06it/s]



✅ Matrice S_KG enrichie calculée
   Shape : (53897, 2812)
   Users trouvés dans RotatE : 52020/53897
   Valeurs NaN : 5,278,124
   Moyenne (hors NaN) : -11.089


In [8]:
import numpy as np

print("Normalisation de la matrice S_KG enrichie...")

# 1. Imputer les NaN par la médiane (comme l'original)
median_val = np.nanmedian(mat_skg_new)
print(f"  Médiane (imputation NaN) : {median_val:.3f}")
mat_skg_new_filled = np.where(np.isnan(mat_skg_new), median_val, mat_skg_new)

# 2. Normalisation Min-Max globale [0,1]
smin = mat_skg_new_filled.min()
smax = mat_skg_new_filled.max()
mat_skg_new_norm = (mat_skg_new_filled - smin) / (smax - smin)
print(f"  Avant norm : [{smin:.2f}, {smax:.2f}]")
print(f"  Après norm : [{mat_skg_new_norm.min():.2f}, {mat_skg_new_norm.max():.2f}]")
print(f"  Moyenne normalisée : {mat_skg_new_norm.mean():.3f}")

# 3. Sauvegarder
RESULTS_V2 = r'D:\Memoire\LLMKGER\kge_results_v2_enriched'
np.save(os.path.join(RESULTS_V2, 'mat_S_KG_enriched_norm.npy'), mat_skg_new_norm)
print(f"\n💾 Matrice S_KG enrichie normalisée sauvegardée")
print(f"   {os.path.join(RESULTS_V2, 'mat_S_KG_enriched_norm.npy')}")

# Comparaison avec l'ancienne S_KG
mat_skg_old = np.load(r'D:\Memoire\LLMKGER\reco_results_final\mat_S_KG_norm.npy', mmap_mode='r')
print(f"\n=== Comparaison ===")
print(f"  S_KG ancienne — moyenne : {np.asarray(mat_skg_old).mean():.3f}")
print(f"  S_KG enrichie — moyenne : {mat_skg_new_norm.mean():.3f}")

Normalisation de la matrice S_KG enrichie...
  Médiane (imputation NaN) : -10.975
  Avant norm : [-23.68, -1.89]
  Après norm : [0.00, 1.00]
  Moyenne normalisée : 0.578

💾 Matrice S_KG enrichie normalisée sauvegardée
   D:\Memoire\LLMKGER\kge_results_v2_enriched\mat_S_KG_enriched_norm.npy

=== Comparaison ===
  S_KG ancienne — moyenne : 0.606
  S_KG enrichie — moyenne : 0.578


In [9]:
import numpy as np, os, math
from collections import defaultdict

RESULTS_REC = r'D:\Memoire\LLMKGER\reco_results_final'
PREREQ_DIR = r'D:\Memoire\LLMKGER\prereq_enriched'

print("Chargement des 3 signaux enrichis...")

# S_emb (inchangé)
mat_semb = np.load(os.path.join(RESULTS_REC, 'mat_S_emb_norm.npy'), mmap_mode='r')
mat_semb = np.asarray(mat_semb)  # charger en RAM

# S_KG enrichi (calculé à l'instant)
mat_skg = mat_skg_new_norm  # déjà en mémoire

# P_prereq enrichi : on doit le recalculer/recharger
# On avait calculé mat_p_new hier — vérifier s'il est sauvegardé
p_enriched_path = os.path.join(PREREQ_DIR, 'mat_P_prereq_enriched_norm.npy')
if os.path.exists(p_enriched_path):
    mat_p = np.load(p_enriched_path)
    print("  P_prereq enrichi rechargé depuis disque")
else:
    print("  ⚠️ P_prereq enrichi pas sauvegardé — il faut le recalculer")
    print("  (on le recalculera si besoin)")

print(f"\n  S_emb : {mat_semb.shape}")
print(f"  S_KG  : {mat_skg.shape}")
print(f"  P_prereq : {'chargé' if os.path.exists(p_enriched_path) else 'à recalculer'}")

Chargement des 3 signaux enrichis...
  ⚠️ P_prereq enrichi pas sauvegardé — il faut le recalculer
  (on le recalculera si besoin)

  S_emb : (53897, 2812)
  S_KG  : (53897, 2812)
  P_prereq : à recalculer


In [10]:
import pandas as pd, os, numpy as np
from collections import defaultdict
from tqdm import tqdm

KG_DIR = r'D:\Memoire\LLMKGER\recbole_data_large\llmkger'
PREREQ_DIR = r'D:\Memoire\LLMKGER\prereq_enriched'

# 1. Charger prérequis enrichis (41 272)
prereq_new = pd.read_csv(os.path.join(PREREQ_DIR, 'prerequisite_of_enriched.csv'), sep='\t')
prereqs_of_concept_new = defaultdict(set)
for h, r, t in prereq_new.itertuples(index=False):
    prereqs_of_concept_new[str(t)].add(str(h))
print(f"Prérequis enrichis : {len(prereq_new):,}")

# 2. concepts_of_course depuis le KG
kg = pd.read_csv(os.path.join(KG_DIR, 'llmkger.kg'), sep='\t')
kg.columns = ['head','relation','tail']
concepts_of_course = defaultdict(set)
for h,t in kg[kg['relation']=='belongs_to'][['head','tail']].itertuples(index=False):
    h,t=str(h),str(t)
    if h.startswith('K') and t.startswith('C_'): concepts_of_course[t].add(h)
    elif t.startswith('K') and h.startswith('C_'): concepts_of_course[h].add(t)

# 3. prereq_of_course enrichi
prereq_of_course_new = {}
for course, concs in concepts_of_course.items():
    req = set()
    for c in concs: req |= prereqs_of_concept_new.get(c, set())
    if req: prereq_of_course_new[course] = req
print(f"Cours avec prérequis (enrichi) : {len(prereq_of_course_new):,}")

# 4. Vérifier qu'on a train_pos et les mappings en mémoire
try:
    print(f"train_pos en mémoire : {len(train_pos):,} users")
    print(f"id2item en mémoire : {len(id2item):,}")
    have_data = True
except NameError:
    print("⚠️ train_pos ou id2item absents — il faut recharger le split RecBole")
    have_data = False

Prérequis enrichis : 41,272
Cours avec prérequis (enrichi) : 575
⚠️ train_pos ou id2item absents — il faut recharger le split RecBole


In [11]:
# Vérifier précisément ce qui est en mémoire
for var in ['id2item', 'item2id', 'candidate_courses', 'users_order', 'train_pos', 'test_item', 'valid_item']:
    try:
        v = eval(var)
        n = len(v)
        print(f"  ✅ {var} : {n:,}")
    except NameError:
        print(f"  ❌ {var} : absent")

  ✅ id2item : 2,813
  ✅ item2id : 2,813
  ✅ candidate_courses : 2,812
  ✅ users_order : 53,897
  ❌ train_pos : absent
  ❌ test_item : absent
  ❌ valid_item : absent


In [13]:
import gc
from recbole.config import Config
from recbole.data import create_dataset, data_preparation
from recbole.utils import init_seed, init_logger
from collections import defaultdict
import numpy as np

RECBOLE_ROOT = r'D:\Memoire\LLMKGER\recbole_data_large'

base_config = {
    'data_path': RECBOLE_ROOT, 'dataset': 'llmkger',
    'USER_ID_FIELD':'user_id','ITEM_ID_FIELD':'item_id','RATING_FIELD':'rating',
    'load_col':{'inter':['user_id','item_id','rating']},
    'eval_args':{'split':{'LS':'valid_and_test'},'order':'RO','group_by':'user','mode':'full'},
    'metrics':['NDCG'],'topk':[10],'valid_metric':'NDCG@10',
    'epochs':1,'device':'cpu','seed':42,'show_progress':False,
}
config = Config(model='LightGCN', dataset='llmkger', config_dict=base_config)
init_seed(config['seed'], config['reproducibility']); init_logger(config)
dataset = create_dataset(config)
train_data, valid_data, test_data = data_preparation(config, dataset)

n_users = dataset.num(config['USER_ID_FIELD'])
user2id = dict(dataset.field2token_id[config['USER_ID_FIELD']])
id2user = {v:k for k,v in user2id.items()}

# Extraire le split
def extract_split(dl):
    pos_item, mask = {}, {}
    for batch in dl:
        interaction, history_index, positive_u, positive_i = batch
        users = interaction[config['USER_ID_FIELD']].numpy()
        pu, pi = positive_u.numpy(), positive_i.numpy()
        for lu, x in zip(pu, pi):
            pos_item[int(users[lu])] = int(x)
        if history_index is not None and len(history_index)==2:
            rows, cols = history_index[0].numpy(), history_index[1].numpy()
            for r,c in zip(rows,cols):
                mask.setdefault(int(users[r]), set()).add(int(c))
    return pos_item, mask

test_item, test_mask = extract_split(test_data)
valid_item, valid_mask = extract_split(valid_data)

train_pos = defaultdict(set)
inter_all = dataset.inter_feat
uu = inter_all[config['USER_ID_FIELD']].numpy()
ii = inter_all[config['ITEM_ID_FIELD']].numpy()
for u,i in zip(uu,ii):
    train_pos[int(u)].add(int(i))
for u in list(train_pos.keys()):
    train_pos[u].discard(test_item.get(u,-1))
    train_pos[u].discard(valid_item.get(u,-1))

print(f"✅ Split rechargé — Test:{len(test_item):,} Valid:{len(valid_item):,} Train users:{len(train_pos):,}")

del train_data, valid_data, test_data, dataset
gc.collect()
print("✅ Mémoire libérée")

c:\Users\AMIRA\anaconda3\envs\llmkger\lib\site-packages\recbole\data\dataset\dataset.py:648: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  feat[field].fillna(value=0, inplace=True)
c:\Users\AMIRA\anaconda3\envs\llmkger\lib\site-packages\recbole\data\dataset\dataset.py:650: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are sett

✅ Split rechargé — Test:53,897 Valid:53,897 Train users:53,897
✅ Mémoire libérée


In [14]:
import numpy as np
from tqdm import tqdm

print("Recalcul de P_prereq enrichi...")

# user externe -> index interne RecBole
uext2idx = {id2user[i]:i for i in range(n_users) if id2user.get(i)}

# concepts couverts par historique train
cov_cache = {}
def covered(u_idx):
    if u_idx in cov_cache: return cov_cache[u_idx]
    cov = set()
    for ci in train_pos.get(u_idx, ()):
        cov |= concepts_of_course.get(id2item.get(ci,''), set())
    cov_cache[u_idx] = cov
    return cov

# cand_index (colonne de chaque cours)
cand_index = {c:j for j,c in enumerate(candidate_courses)}
N = len(users_order)
M = len(candidate_courses)

# Matrice P_prereq enrichi
mat_p_new = np.zeros((N, M), dtype=np.float32)
n_active = 0
for j, course in enumerate(tqdm(candidate_courses)):
    req = prereq_of_course_new.get(course)
    if not req: continue
    n_active += 1
    for row, u_ext in enumerate(users_order):
        u_idx = uext2idx.get(u_ext)
        if u_idx is None: continue
        cov = covered(u_idx)
        nc = len(req - cov)
        if nc > 0:
            mat_p_new[row, j] = nc / len(req)

# Normaliser [0,1]
pmin, pmax = mat_p_new.min(), mat_p_new.max()
mat_p_new_norm = (mat_p_new - pmin) / (pmax - pmin) if pmax > pmin else mat_p_new

# Sauvegarder
import os
PREREQ_DIR = r'D:\Memoire\LLMKGER\prereq_enriched'
np.save(os.path.join(PREREQ_DIR, 'mat_P_prereq_enriched_norm.npy'), mat_p_new_norm)

print(f"\n✅ P_prereq enrichi : {mat_p_new_norm.shape}")
print(f"   Cours actifs : {n_active}")
print(f"   Moyenne : {mat_p_new_norm.mean():.4f}")
print(f"   💾 Sauvegardé : mat_P_prereq_enriched_norm.npy")

Recalcul de P_prereq enrichi...


100%|██████████| 2812/2812 [17:16<00:00,  2.71it/s] 



✅ P_prereq enrichi : (53897, 2812)
   Cours actifs : 575
   Moyenne : 0.1808
   💾 Sauvegardé : mat_P_prereq_enriched_norm.npy


In [15]:
import numpy as np, os, math

RESULTS_REC = r'D:\Memoire\LLMKGER\reco_results_final'

# Charger S_emb (inchangé)
mat_semb = np.asarray(np.load(os.path.join(RESULTS_REC, 'mat_S_emb_norm.npy'), mmap_mode='r'))

# S_KG enrichi et P_prereq enrichi (déjà en mémoire)
mat_skg = mat_skg_new_norm
mat_p = mat_p_new_norm

print("Signaux enrichis prêts :")
print(f"  S_emb : {mat_semb.shape}")
print(f"  S_KG  : {mat_skg.shape}")
print(f"  P_prereq : {mat_p.shape}")

# Colonnes cibles + masques (test & valid)
test_col = np.full(N, -1, dtype=np.int64)
valid_col = np.full(N, -1, dtype=np.int64)
mask_test = np.zeros((N, M), dtype=bool)
mask_valid = np.zeros((N, M), dtype=bool)

for row, u_ext in enumerate(users_order):
    u_idx = uext2idx.get(u_ext)
    if u_idx is None: continue
    tc = id2item.get(test_item.get(u_idx, -1))
    test_col[row] = cand_index.get(tc, -1)
    for c in test_mask.get(u_idx, ()):
        jm = cand_index.get(id2item.get(c))
        if jm is not None: mask_test[row, jm] = True
    vc = id2item.get(valid_item.get(u_idx, -1))
    valid_col[row] = cand_index.get(vc, -1)
    for c in valid_mask.get(u_idx, ()):
        jm = cand_index.get(id2item.get(c))
        if jm is not None: mask_valid[row, jm] = True

print(f"\nCours test mappés : {(test_col>=0).sum():,}/{N}")
print(f"Cours valid mappés : {(valid_col>=0).sum():,}/{N}")

Signaux enrichis prêts :
  S_emb : (53897, 2812)
  S_KG  : (53897, 2812)
  P_prereq : (53897, 2812)

Cours test mappés : 53,897/53897
Cours valid mappés : 53,897/53897


In [16]:
import numpy as np, math, time
from skopt import gp_minimize
from skopt.space import Real

def ndcg_at(score_mat, target_col, mask, k=10):
    s = score_mat.astype(np.float64).copy()
    s[mask] = -1e18
    tot = 0.0; v = 0
    for i in range(len(target_col)):
        j = target_col[i]
        if j < 0: continue
        target = s[i, j]
        greater = int((s[i] > target).sum())
        ties = int((s[i] == target).sum())
        rank = greater + (ties+1)/2.0
        if rank <= k: tot += 1.0/math.log2(rank+1)
        v += 1
    return tot/v if v else 0.0

def full_metrics(score_mat, target_col, mask, ks=(5,10)):
    s = score_mat.astype(np.float64).copy()
    s[mask] = -1e18
    res = {f'{m}@{k}':[] for k in ks for m in ['ndcg','mrr','recall']}
    for i in range(len(target_col)):
        j = target_col[i]
        if j < 0: continue
        target = s[i, j]
        greater = int((s[i] > target).sum())
        ties = int((s[i] == target).sum())
        rank = greater + (ties+1)/2.0
        for k in ks:
            hit = 1.0 if rank<=k else 0.0
            res[f'recall@{k}'].append(hit)
            res[f'ndcg@{k}'].append(1.0/math.log2(rank+1) if rank<=k else 0.0)
            res[f'mrr@{k}'].append(1.0/rank if rank<=k else 0.0)
    return {m:float(np.mean(v)) for m,v in res.items()}

# Bayesian Optimization sur VALIDATION (3 signaux enrichis)
def objective(params):
    a, b = params
    if a + b > 1.0: return 0.0
    l = 1.0 - a - b
    score = a*mat_skg + b*mat_semb - l*mat_p
    return -ndcg_at(score, valid_col, mask_valid, k=10)

print("Bayesian Optimization (S_KG enrichi + S_emb + P_prereq enrichi)...")
t0 = time.time()
res_bo = gp_minimize(objective, [Real(0,1,name='a'), Real(0,1,name='b')],
                     n_calls=40, n_initial_points=12, random_state=42)
a_opt, b_opt = res_bo.x
l_opt = 1.0 - a_opt - b_opt
print(f"Terminé en {(time.time()-t0)/60:.1f} min")
print(f"\n🏆 Poids optimaux : α={a_opt:.4f} β={b_opt:.4f} λ={l_opt:.4f}")
print(f"   NDCG@10 valid : {-res_bo.fun:.4f}")

Bayesian Optimization (S_KG enrichi + S_emb + P_prereq enrichi)...
Terminé en 1.3 min

🏆 Poids optimaux : α=0.4158 β=0.5624 λ=0.0219
   NDCG@10 valid : 0.1106


In [17]:
import numpy as np, json, os

# Évaluation sur TEST avec les poids optimaux (3 signaux enrichis)
score_final = a_opt*mat_skg + b_opt*mat_semb - l_opt*mat_p
res_test_enriched = full_metrics(score_final, test_col, mask_test)

# Comparaison avec le pipeline original (P_prereq ancien + S_KG ancien)
mat_skg_old = np.asarray(np.load(os.path.join(RESULTS_REC, 'mat_S_KG_norm.npy'), mmap_mode='r'))
mat_p_old = np.asarray(np.load(os.path.join(RESULTS_REC, 'mat_P_prereq_norm.npy'), mmap_mode='r'))
score_old = 0.607*mat_skg_old + 0.334*mat_semb - 0.059*mat_p_old
res_test_old = full_metrics(score_old, test_col, mask_test)

print("="*70)
print("  COMPARAISON FINALE — Pipeline ORIGINAL vs CHAÎNE ENRICHIE")
print("="*70)
print(f"{'Métrique':<14}{'Original':>16}{'Enrichi (L4)':>16}{'Δ':>12}")
print("-"*70)
for k in ['ndcg@10','mrr@10','recall@10','ndcg@5','mrr@5','recall@5']:
    o = res_test_old[k]; e = res_test_enriched[k]
    print(f"{k:<14}{o:>16.4f}{e:>16.4f}{e-o:>+12.4f}")
print("="*70)
print(f"\nPoids original : α=0.607 β=0.334 λ=0.059")
print(f"Poids enrichi  : α={a_opt:.3f} β={b_opt:.3f} λ={l_opt:.3f}")

# Sauvegarde
PREREQ_DIR = r'D:\Memoire\LLMKGER\prereq_enriched'
out = {'original':res_test_old, 'enrichi_complet':res_test_enriched,
       'poids':{'alpha':a_opt,'beta':b_opt,'lambda':l_opt},
       'valid_ndcg':-res_bo.fun}
with open(os.path.join(PREREQ_DIR, 'reeval_chaine_complete.json'),'w') as f:
    json.dump(out, f, indent=2)
print("\n💾 Résultats sauvegardés : reeval_chaine_complete.json")

  COMPARAISON FINALE — Pipeline ORIGINAL vs CHAÎNE ENRICHIE
Métrique              Original    Enrichi (L4)           Δ
----------------------------------------------------------------------
ndcg@10                 0.1037          0.1119     +0.0082
mrr@10                  0.0919          0.0943     +0.0024
recall@10               0.1427          0.1699     +0.0271
ndcg@5                  0.0939          0.0978     +0.0039
mrr@5                   0.0879          0.0886     +0.0006
recall@5                0.1123          0.1261     +0.0139

Poids original : α=0.607 β=0.334 λ=0.059
Poids enrichi  : α=0.416 β=0.562 λ=0.022

💾 Résultats sauvegardés : reeval_chaine_complete.json


In [2]:
import matplotlib.pyplot as plt
import numpy as np
import os

# ── Données vérifiées (04_kge_training.ipynb, outputs réels) ──
models = ['TransE', 'RotatE', 'ComplEx', 'DistMult']
metrics = ['MRR', 'Hit@1', 'Hit@3', 'Hit@10']

values = {
    'MRR'   : [0.0734, 0.2012, 0.0171, 0.0007],
    'Hit@1' : [0.0153, 0.1406, 0.0079, 0.0004],
    'Hit@3' : [0.0929, 0.2114, 0.0166, 0.0007],
    'Hit@10': [0.1755, 0.3174, 0.0334, 0.0010],
}

# ── Couleurs ──
colors = ['#7EC8E3', '#E63946', '#F4A261', '#B5EAD7']

# ── Figure ──
x = np.arange(len(metrics))
width = 0.18
fig, ax = plt.subplots(figsize=(10, 6))

for i, model in enumerate(models):
    vals = [values[m][i] for m in metrics]
    bars = ax.bar(x + i * width, vals, width,
                  label=model, color=colors[i],
                  edgecolor='black', linewidth=0.5)
    # Valeur au-dessus de chaque barre
    for bar, v in zip(bars, vals):
        if v > 0.001:
            ax.text(bar.get_x() + bar.get_width()/2,
                    bar.get_height() + 0.003,
                    f'{v:.3f}', ha='center', va='bottom',
                    fontsize=7.5, fontweight='bold')

# ── Mise en forme ──
ax.set_xlabel('Metric', fontsize=12)
ax.set_ylabel('Score', fontsize=12)
ax.set_title('Comparison of Knowledge Graph Embedding Models', fontsize=13, fontweight='bold')
ax.set_xticks(x + width * 1.5)
ax.set_xticklabels(metrics, fontsize=11)
ax.set_ylim(0, 0.38)
ax.legend(fontsize=10, loc='upper left')
ax.yaxis.grid(True, linestyle='--', alpha=0.7)
ax.set_axisbelow(True)

plt.tight_layout()
plt.savefig('kge_comparison.png', dpi=200, bbox_inches='tight')
plt.close()
print("kge_comparison.png sauvegardé dans :", os.getcwd())

kge_comparison.png sauvegardé dans : d:\Memoire\LLMKGER\notebooks
